In [ ]:
import matplotlib
print("Versione:", matplotlib.__version__)
print("Percorso:", matplotlib.__file__)

In [ ]:
# ==========================================================
# GEOPHPY - PROCESSING MAGNETIC Optimazed data 
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np
import copy     # <--- serve per le copie interne ai filtri
from geophpy.dataset import DataSet
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# GeophPy 0.32.2 

# =====================================
# Import file
# =====================================
file_path = "path/of/your/xyz/file/file.XYZ"


# plot first 20 rows (for check)
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(20):
        line = f.readline()
        print(f"{i:02d}: {line.strip()}")

In [ ]:
# """""""""""""""""""""""""""
# 
# """""""""""""""""""""""""""

from geophpy.dataset import DataSet

# Grad
success, grad_ds = DataSet.from_file([file_path], fileformat='ascii',
                                     delimiter=' ', x_colnum=1, y_colnum=2, z_colnum=3)

# Total field

#success, tot_ds = DataSet.from_file([file_path], fileformat='ascii',
                                    #delimiter=' ', x_colnum=1, y_colnum=2, z_colnum=4)

print("locaded?", success)



In [ ]:
print("Tipo oggetto:", type(grad_ds))
#print("Type object:", type(tot_ds))

In [ ]:
# === Threshold e peak filtering ===
grad_ds.threshold(setmin=-10, setmax=10)
grad_ds.peakfilt(method='median', halfwidth=5, threshold=15, mode='absolute')

In [ ]:
import copy

grad_peak = copy.deepcopy(grad_ds)

In [ ]:
# === Manual Plot ===
grad_peak.plot(
    plottype='2D-SCATTER',
    cmmin=-10,
    cmmax=10,
    cmapname='gray',
    filename='Geomagnetic_Map_after_Threshold_and_peakfilt.png',
    interpolation='cubic',    # solo effetto visivo
    transparent=True,
    dpi=400
)
# Save the histogram as a temporary image
grad_peak.histo_plot(
    zmin=-10, zmax=10,
    cmapdisplay=True,
    coloredhisto=True
)
plt.savefig("temp_hist.png", dpi=400, bbox_inches='tight')
plt.close()

# === It combines a map and a histogram in a single figure ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, (ax1, ax2) = plt.subplots(
    ncols=2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [4, 1]}
)

# Load the map that has just been generated
img_map = mpimg.imread('Geomagnetic_Map_after_Threshold_and_peakfilt.png')
ax1.imshow(img_map)
ax1.axis('off')
ax1.set_title("Rocavecchia – Geomagnetic Map after Threshold and Peak Filtering", fontsize=12)

# Load the histogram
img_hist = mpimg.imread('temp_hist.png')
ax2.imshow(img_hist)
ax2.axis('off')
ax2.set_title("Histogram", fontsize=10)

plt.tight_layout()

# === Final export ===
fig.savefig("Rocavecchia_geomagnetic_Threshold.png", dpi=600, bbox_inches='tight')
fig.savefig("Rocavecchia_geomagnetic_Threshold.pdf", dpi=600, bbox_inches='tight')

plt.show()

# (Optional) Delete temporary files
import os
os.remove('temp_hist.png')


In [ ]:
# === Destaggering (festoon correction) ===
# grad_ds.festoonfilt(method='Crosscorr', uniformshift=True)

In [ ]:
# === Destriping (costant) ===
grad_ds.destripecon(Nprof='all', method='additive', config='mono', reference='median')

In [ ]:
# === Zero-mean traverse (optional for single-sensor systems) ===
grad_ds.zeromeanprofile(setvar='median')

In [ ]:
grad_destripe_zeromean = copy.deepcopy(grad_ds)

In [ ]:
# === You can now use the diagram from the manual ===
grad_destripe_zeromean.plot(
    plottype='2D-SCATTER',
    cmmin=-10,
    cmmax=10,
    cmapname='gray_r',
    filename='Geomagnetic_Map_after_Threshold_and_peakfilt.png',
    interpolation='cubic',   
    transparent=True,
    dpi=400
)

zmin, zmax = grad_destripe_zeromean.histo_getlimits()

grad_destripe_zeromean.histo_plot(zmin=-10, zmax=10, cmapdisplay=True, coloredhisto=True)

In [ ]:
grad_destripe_zeromean.plot(
    plottype='2D-SCATTER',
    cmmin=-10,
    cmmax=10,
    cmapname='gray',
    filename='Geomagnetic_Map_after_Destripe_and_Zero-mean.png',  # ✅ corretto
    interpolation='cubic',
    transparent=True,
    dpi=400
)

# === Calculating limits and generating a histogram ===
zmin, zmax = grad_destripe_zeromean.histo_getlimits()

grad_destripe_zeromean.histo_plot(
    zmin=-10, zmax=10,
    cmapdisplay=True,
    coloredhisto=True
)
plt.savefig("temp_hist.png", dpi=400, bbox_inches='tight')
plt.close()

# === It combines a map and a histogram in a single figure  ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, (ax1, ax2) = plt.subplots(
    ncols=2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [4, 1]}
)

#  Load the map that has just been generated
img_map = mpimg.imread('Geomagnetic_Map_after_Destripe_and_Zero-mean.png')
ax1.imshow(img_map)
ax1.axis('off')
ax1.set_title("Rocavecchia – Geomagnetic Map after Destripe and Zero-mean Filtering", fontsize=12)

# Load the histogram
img_hist = mpimg.imread('temp_hist.png')
ax2.imshow(img_hist)
ax2.axis('off')
ax2.set_title("Histogram", fontsize=10)

plt.tight_layout()

# === Final export ===
fig.savefig("Rocavecchia_geomagnetic_Destripe.png", dpi=600, bbox_inches='tight')
fig.savefig("Rocavecchia_geomagnetic_Destripe.pdf", dpi=600, bbox_inches='tight')

plt.show()

# (Optional) Delete temporary files
import os
os.remove('temp_hist.png')


In [ ]:
# Optional

#grad_ds.ploughfilt(apod=0, azimuth=255, cutoff=80, width=5)

In [ ]:
# === Normalization filter ===
#grad_ds.medianfilt(nx=5, ny=5)

grad_ds.interpolate(interpolation='linear')

grad_ds.wallisfilt(
    nx=9, ny=9,
    targmean=120,
    targstdev=60,
    setgain=4,
    limitstdev=20,
    edgefactor=0.3
)

In [ ]:
grad_wallis = copy.deepcopy(grad_ds)

In [ ]:
# === Manual plot ===

grad_ds.plot(
    plottype='2D-SURFACE',
    #cmmin=-10,
    #cmmax=10,
    cmapname='gray',
    filename='Geomagnetic_Map_after_Wallis_Filter.png',  # ✅ nome aggiornato
    transparent=True,
    dpi=400
)

# === Calculating limits and generating a histogram ===
zmin, zmax = grad_wallis.histo_getlimits()

grad_ds.histo_plot(
    zmin=-10, zmax=10,
    cmapdisplay=True,
    coloredhisto=True
)
plt.savefig("temp_hist.png", dpi=400, bbox_inches='tight')
plt.close()

# === Combines a map and a histogram in a single figure ===
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, (ax1, ax2) = plt.subplots(
    ncols=2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [4, 1]}
)

# Load the map that has just been generated
img_map = mpimg.imread('Geomagnetic_Map_after_Wallis_Filter.png')
ax1.imshow(img_map)
ax1.axis('off')
ax1.set_title("Rocavecchia – Geomagnetic Map after Wallis Filtering", fontsize=12)

# Load Histogram
img_hist = mpimg.imread('temp_hist.png')
ax2.imshow(img_hist)
ax2.axis('off')
ax2.set_title("Histogram", fontsize=10)

plt.tight_layout()

# === Final export ===
fig.savefig("Rocavecchia_geomagnetic_Wallis.png", dpi=600, bbox_inches='tight')
fig.savefig("Rocavecchia_geomagnetic_Wallis.pdf", dpi=600, bbox_inches='tight')

plt.show()

# (Optional) remove temporary files
import os
os.remove('temp_hist.png')


In [ ]:
# === Final plot ===

zmin, zmax = grad_ds.histo_getlimits()
fig, cmap = grad_ds.plot(plottype='2D-SURFACE',
                         cmapname='gray',
                         transparent=True, dpi=600)
plt.title("Rocavecchia – Gradiente filtrato")
plt.show()

grad_ds.histo_plot(cmapdisplay=True, coloredhisto=True)
plt.show()

In [ ]:
# === Ensure the dataset is gridded ===
# (required for correlation)
#grad_ds.interpolate(interpolation=“linear”)

# ===  Calculate and save the Crosscorr correlation map ===
grad_ds.correlation_plotmap(
    method="Crosscorr",
    filename="Rocavecchia_Correlation_Map.png",  # ✅ esporta la mappa
    dpi=600,                                     # alta risoluzione
    transparent=True,                            # sfondo trasparente (opzionale)
    cmapname='jet',                              # colormap tipica per correlazioni
    cmapdisplay=True                             # mostra colorbar
)

# === Show the figure (optional) ===
import matplotlib.pyplot as plt
plt.show()

# === (Optional) Export to PDF as well ===
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

img_corr = mpimg.imread('Rocavecchia_Correlation_Map.png')
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(img_corr)
ax.axis('off')
ax.set_title("Rocavecchia – Profile Cross-Correlation Map", fontsize=12)

fig.savefig("Rocavecchia_Correlation_Map.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# === Make sure the dataset is on a grid ===

# === Calculate and save the correlation sum map ===
grad_ds.correlation_plotsum(filename="Rocavecchia_Correlation_SumMap.png",  # ✅ salva la mappa                            # mostra la colorbar
)

# === Show figure (Optional) ===
import matplotlib.pyplot as plt
plt.show()

# === Export PDF (Optional) ===
import matplotlib.image as mpimg

img_sum = mpimg.imread('Rocavecchia_Correlation_SumMap.png')
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(img_sum)
ax.axis('off')
ax.set_title("Rocavecchia – Profile Correlation Sum Map", fontsize=12)

fig.savefig("Rocavecchia_Correlation_SumMap.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# === Mean Track Plot (only raw data) ===
grad_ds.meantrack_plot(
    Nprof=4,                     # numero di profili mediati (finestra)
    method='additive',           # tipo di correzione offset
    Ndeg=None,                   # opzionale: usa None per profili lineari
    plotflag='raw',              # mostra solo i dati grezzi
    filename='Rocavecchia_Mean_cross-track.png',  # ✅ nome file esportato
)

# === Show plot ===
import matplotlib.pyplot as plt
plt.show()

# === Export PDF (Optional) ===
import matplotlib.image as mpimg

img_mean = mpimg.imread('Rocavecchia_Mean_cross-track.png')
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_mean)
ax.axis('off')
ax.set_title("Rocavecchia – Mean Cross-Track (Raw Profiles)", fontsize=12)

fig.savefig("Rocavecchia_Mean_cross-track.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# === Mean Track Plot (raw + corrected) ===
grad_ds.meantrack_plot(
    Nprof=4,
    method='additive',
    Ndeg=None,
    plotflag='destriped',                    # ✅ mostra profili raw + corretti
    filename='Rocavecchia_Mean_cross-track_both.png',
)



plt.title("Mean cross-track profile", pad=20)   # sposta verso l’alto la scritta
plt.tight_layout(pad=2.5)                       # sistema automaticamente margini
plt.show()


# === Show plot ===

plt.show()

# === Export PDF ===
import matplotlib.image as mpimg

img_mean_both = mpimg.imread('Rocavecchia_Mean_cross-track_both.png')
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_mean_both)
ax.axis('off')
ax.set_title("Rocavecchia – Mean Cross-Track (Raw + Corrected)", fontsize=12)

fig.savefig("Rocavecchia_Mean_cross-track_both.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
from scipy import stats
from scipy.signal import correlate2d

In [ ]:
Z = grad_ds.data.z_image
Z = Z[~np.isnan(Z)]

print("Totale punti:", len(Z))
print("Media:", np.mean(Z))
print("Mediana:", np.median(Z))
print("Deviazione standard:", np.std(Z))
print("Asimmetria (skewness):", stats.skew(Z))
print("Curtosi:", stats.kurtosis(Z))

In [ ]:
# === 1️⃣ Istogramma ===
plt.figure(figsize=(6,4))
plt.hist(Z, bins=100, color='gray')
plt.title("Distribuzione valori del gradiente filtrato")
plt.xlabel("Intensità (nT/m)")
plt.ylabel("Frequenza")
plt.show()